Датасет :[Telco Customer Churn на Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn).

Задача: по данным об услугах, договоре и платежах предсказываем, ушёл ли клиент

In [1]:
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

url = "https://www.kaggle.com/api/v1/datasets/download/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"
data = pd.read_csv(url, na_values=" ")
data[["tenure", "Contract", "MonthlyCharges", "TotalCharges", "Churn"]].head()

,tenure,Contract,MonthlyCharges,TotalCharges,Churn
0,1,Month-to-month,29.85,29.85,No
1,34,One year,56.95,1889.50,No
2,2,Month-to-month,53.85,108.15,Yes
3,45,One year,42.30,1840.75,No
4,2,Month-to-month,70.70,151.65,Yes


In [2]:
X = data.drop(columns=["customerID", "Churn"])
y = data["Churn"].eq("Yes").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_columns = X_train.select_dtypes(include="number").columns
categorical_columns = X_train.select_dtypes(exclude="number").columns

preprocessing = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_columns),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
])

forest = RandomForestClassifier(max_depth=8, class_weight="balanced", random_state=42)
model = make_pipeline(preprocessing, forest)
model.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('numeric',
                                                  SimpleImputer(strategy='median'),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                ('randomforestclassifier',
                 RandomForestClassifier(class_weight='balanced', max_depth=8,
                                        random_state=42))])

In [3]:
predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, predictions),
    "f1": f1_score(y_test, predictions),
    "roc_auc": roc_auc_score(y_test, probabilities),
}

print(f"Клиентов в датасете: {len(data)}")
print(f"Клиентов в тестовой выборке: {len(y_test)}")
print(f"Accuracy: {metrics['accuracy']:.3f}")
print(f"F1: {metrics['f1']:.3f}")
print(f"ROC-AUC: {metrics['roc_auc']:.3f}")

model.validation_metrics_ = metrics
joblib.dump(model, "model.joblib", compress=3)

Клиентов в датасете: 7043
Клиентов в тестовой выборке: 1409
Accuracy: 0.762
F1: 0.632
ROC-AUC: 0.843


['model.joblib']